# GUMELAB · Reconocimiento de entidades nombradas (NER)
Extrae personas, lugares y organizaciones de los textos en español con `es_core_news_lg` de spaCy. No requiere GPU (≈ 20–25 min en CPU).

Entrada: `textos_gumelab.csv.gz` (generado por `scripts/01_exportar_textos.py`) en `MyDrive/GUMELAB_sentimiento/`.
Salida: `entidades_gumelab.zip` → descomprimir en `datos_privados/derivados/`.

In [ ]:
!pip -q install -U spacy
!python -m spacy download es_core_news_lg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, re, pandas as pd
CARPETA = '/content/drive/MyDrive/GUMELAB_sentimiento'
os.makedirs(CARPETA + '/ner_bloques', exist_ok=True)
f = [x for x in glob.glob(CARPETA + '/textos_gumelab*') if x.endswith(('.csv', '.zip', '.csv.gz'))][0]
d = pd.read_csv(f, dtype={'id': str})
d = d[d.texto.notna()]; d['texto'] = d.texto.astype(str)
ES = re.compile(r'\b(?:que|de|la|el|en|los|las|por|para|con|una|es|muy|pero|como|más|serie|mejor)\b', re.I)
d = d[d.texto.str.contains(ES)].reset_index(drop=True)
d['limpio'] = (d.texto.str.replace(r'http\S+|www\.\S+', ' ', regex=True)
                      .str.replace(r'@\w+', ' ', regex=True)
                      .str.replace('#', ' ', regex=False)
                      .str.replace(r'\s+', ' ', regex=True).str.strip())
d['clave'] = range(len(d))
print(len(d), 'textos en español')

In [ ]:
import spacy
nlp = spacy.load('es_core_news_lg', exclude=['parser', 'lemmatizer', 'attribute_ruler', 'morphologizer', 'senter'])
print(nlp.pipe_names)
doc = nlp('Pablo Escobar mandó a matar a Luis Carlos Galán en Soacha en 1989, y Caracol hizo la serie.')
print([(e.text, e.label_) for e in doc.ents])

In [ ]:
import time
BLOQUE = 20000; nb = (len(d) + BLOQUE - 1) // BLOQUE; t0 = time.time()
for b in range(nb):
    f = f'{CARPETA}/ner_bloques/n{b:03d}.csv.gz'
    if os.path.exists(f): continue
    q = d.iloc[b*BLOQUE:(b+1)*BLOQUE]
    filas = []
    for clave, doc in zip(q.clave, nlp.pipe(q.limpio.tolist(), batch_size=500, n_process=2)):
        for e in doc.ents:
            if e.label_ in ('PER', 'LOC', 'ORG', 'MISC') and len(e.text) > 2:
                filas.append((clave, e.text, e.label_))
    pd.DataFrame(filas, columns=['clave', 'entidad', 'tipo']).to_csv(f, index=False, compression='gzip')
    print(f'bloque {b+1}/{nb}  {(time.time()-t0)/60:.1f} min')
print('Listo')

In [ ]:
import zipfile
e = pd.concat([pd.read_csv(x) for x in sorted(glob.glob(CARPETA + '/ner_bloques/*.csv.gz'))])
e = e.merge(d[['clave', 'id', 'fuente', 'serie']], on='clave').drop(columns='clave')
e.to_csv('/content/entidades_gumelab.csv', index=False)
with zipfile.ZipFile(CARPETA + '/entidades_gumelab.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/entidades_gumelab.csv', 'entidades_gumelab.csv')
print(len(e), 'menciones de entidades')
from google.colab import files
files.download(CARPETA + '/entidades_gumelab.zip')